In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

from typing import List

In [ ]:
sns.set_style("ticks")
sns.set_context("paper")

In [ ]:
# Set Style
def set_style(ax: plt.Axes, xlabel: str, ylabel: str, title: str = '', xlog: bool = False, xlog_base: float = 1,
              ylog: bool = False, xtick_size: int = 26):
    """
    Set style for matplotlib axes.
    """
    ax.set_xlabel(xlabel, fontsize=32, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=30, labelpad=10)
    if title: ax.set_title(title, fontsize=36)
    if xlog:
        if xlog_base == 1: ax.set_xscale('log')
        else: ax.set_xscale('log', base=xlog_base)
    if ylog: ax.set_yscale('log')
    ax.tick_params(axis='x', labelsize=xtick_size)
    ax.tick_params(axis='y', labelsize=26)
    # despine ax
    sns.despine(ax=ax)

In [ ]:
# -- plot over aggregated trials
def plot_over_trials(exact_df: pd.DataFrame, cum_apx_df: pd.DataFrame, cum_rh_df: pd.DataFrame, axes: List[plt.Axes],
                     plot_type: str = 'NDCC', bin_size: float = 2) -> None:
    ax_distro = axes[0]

    exact_style = {"markersize": 7.5, "color": 'blue', "marker": 'o', "markeredgewidth": .05, "markeredgecolor": 'black', 'linewidth': 2}
    apx_style = {"markersize": 15, "errorbar": 'sd', "err_style": "bars", "zorder": 10, "color": 'orange',
                 "marker": 'x', "markeredgewidth": 2, "markeredgecolor": "orange", "linewidth": 0.0,
                 "err_kws": {"capsize": 2, "elinewidth": .5, "capthick": .5}}

    rh_style = {"markersize": 7.5, "errorbar": ('ci', 95), "zorder": 10, "err_style": "band", 'linewidth': 3.5,
                "color": 'green', "marker": 's', "markeredgewidth": .5, "markeredgecolor": 'black'}

    # plot exact vs est
    exact_df = exact_df[exact_df['binned_id'] > 0]
    cum_apx_df = cum_apx_df[cum_apx_df['binned_id'] > 0]
    # let's discard cum_apx_df measures with zero node count for cleaner plots (these are typically very noisy and not meaningful)
    cum_apx_df = cum_apx_df[cum_apx_df['est_node_count'] > 0]
    y_show_exact = 'ndcc' if 'NDCC' in plot_type else 'wdcc'
    y_show_apx = 'est_ndcc' if 'NDCC' in plot_type else 'est_wdcc'
    sns.lineplot(data=exact_df, ax=ax_distro, x='binned_id', y=y_show_exact, legend=False, **exact_style)
    sns.lineplot(data=cum_apx_df, ax=ax_distro, x='binned_id', y=y_show_apx, legend=False, **apx_style)
    ylabel = 'WDCC' if 'WDCC' in plot_type else 'NDCC'

    # # set style
    xtick_size = 32 if bin_size == 2 else 26
    set_style(ax_distro, xlabel='Degree', ylabel=ylabel, title=dataset_name, xlog=False, ylog=False, xtick_size=xtick_size)
    # set fancy xlabels: offset x-ticks to show bin ranges
    min_k, max_k = 2, max(exact_df["binned_id"].max(), cum_apx_df["binned_id"].max())
    step = max(1, int(np.ceil((max_k - min_k) / 4)))
    ks = np.arange(min_k, max_k + 1, step)
    if ks[-1] < (max_k - 1): ks = np.append(ks, max_k)  # always include the rightmost tick
    # # set ylabels: 4 ticks
    y_max = ax_distro.get_ylim()[1]
    y_ticks = np.linspace(0, y_max, num=4)
    ax_distro.set_yticks(y_ticks)
    y_tick_labels = [f'{y:.2f}' for y in y_ticks]
    ax_distro.set_yticklabels(y_tick_labels)

    ax_distro.set_xticks(ks)
    ax_distro.set_xticklabels([fr"${bin_size}^{{{k}}}$" for k in ks])


    ax_rel_h = axes[1]
    # compute relative hausdorff distance
    sns.lineplot(data=cum_rh_df, ax=ax_rel_h, x='binned_id', y='rh_distance', **rh_style)
    # set style
    title_right = ""
    ylabel = 'RHAS'
    set_style(ax_rel_h, xlabel='Degree', ylabel=ylabel, title=title_right, xlog=False, ylog=False, xtick_size=xtick_size)
    ax_rel_h.set_xticks(ks)
    ax_rel_h.set_xticklabels([fr"${bin_size}^{{{k}}}$" for k in ks])

    # set ylim
    avg_per_bin = cum_rh_df.groupby('binned_id')['rh_distance'].mean()
    y_top = avg_per_bin.max()
    y_bottom = avg_per_bin.min()
    y_range = y_top - y_bottom

    y_top = y_top + y_range * 0.3
    y_bottom = y_bottom - y_range * 0.1

    ax_rel_h.set_ylim(bottom=y_bottom, top=y_top)

    # force yticks from y_bottom to y_top
    y_ticks = np.linspace(0, y_top, num=4)
    ax_rel_h.set_yticks(y_ticks)
    # set label rounded to 2 or 3 decimal places
    if y_top < 0.04: y_tick_labels = [f'{y:.3f}' for y in y_ticks]
    else: y_tick_labels = [f'{y:.2f}' for y in y_ticks]
    ax_rel_h.set_yticklabels(y_tick_labels)

In [ ]:
# main driver
dataset_name = 'livejournal'

n_rows, n_cols = 2, 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 4 * n_rows), squeeze=True)

plot_type = 'NDCC'
exact_df = pd.read_csv(f'../res/{dataset_name}_exact_distros.csv')
est_df = pd.read_csv(f'../res/{dataset_name}_est_distros.csv')
rhas_ndcc_df = pd.read_csv(f'../res/{dataset_name}_rhas_ndcc.csv')
rhas_wdcc_df = pd.read_csv(f'../res/{dataset_name}_rhas_wdcc.csv')

plot_over_trials(exact_df, est_df, rhas_ndcc_df, axes, plot_type)

# build legend manually
legend_elements = [
    Line2D([0], [0], marker='o', color='blue', label='Exact', markersize=16, linewidth=3, markeredgewidth=.025, markeredgecolor='black'),
    Line2D([0], [0], marker='x', color='orange', label=f'$\\mathtt{{BOLIDE}}$', markersize=16, markeredgewidth=2, markeredgecolor='orange', linewidth=0.0),
    Line2D([0], [0], marker='s', color='green', label='RHAS Distance', markersize=16, markeredgewidth=0.5, markeredgecolor='black', linewidth=2.5)
]
fig.legend(handles=legend_elements, ncol=4, fontsize=20, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.075));
fig.tight_layout()
